In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import pandas as pd
from pymongo import MongoClient
import yaml
from tqdm import tqdm

pd.set_option('display.max_columns', None)


In [ ]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]

In [ ]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]

In [ ]:
all_games = list(db[mongo_config.get('collections').get('collection_processed_events')].distinct('game_id', {}))

In [ ]:
sequence_action_types = {
    "Pass",
    "TakeOn",
    "BallRecovery",
    "Interception",
    "Tackle",
    "Aerial",
    "BallTouch",
    "GoodSkill",
}

shot_types = {
    "Goal",
    "SavedShot",
    "MissedShots",
    "ShotOnPost",
}

ignore_types = {
    "Save",
    "Claim",
    "Punch",
    "KeeperPickup",
    "KeeperSweeper",
    "SubstitutionOn",
    "SubstitutionOff",
    "Card",
    "Smother",
}

break_types = {
    "OffsideGiven",
    "OffsideProvoked",
    "OffsidePass",
    "Foul",
}

opponent_pressure_types = {
    "Aerial",
    "BallTouch",
    "Challenge",
    "Clearance",
    "Tackle",
    "TakeOn",
}




In [ ]:
def shooting_valid_sequence_event(event, shooting_team_id, current_period):
    event_type = event["type"]
    outcome = event.get("outcome_type")
    event_period = event.get("period")

    if event_period != current_period:
        return "break"
    
    if event_type in ignore_types:
        return "ignore"

    if event_type in break_types:
        return "break"

    same_team = event["team_id"] == shooting_team_id

    if not same_team:
        if event_type == "Error":
            return "keep"
        if event_type in opponent_pressure_types and outcome == "Unsuccessful":
            return "keep"
        return "break"

    if event_type in {"Dispossessed", "Turnover", "Error"}:
        return "break"

    if event_type in {"Pass", "TakeOn", "BallTouch"}:
        return "keep" if outcome == "Successful" else "break"

    if event_type in sequence_action_types:
        return "break" if outcome == "Unsuccessful" else "keep"

    return "ignore"

def get_shot_sequence(game_data, shot_index, sequence_id):
    shooting_team_id = game_data.loc[shot_index, "team_id"]
    current_period = game_data.loc[shot_index, "period"]
    sequence_rows = [game_data.loc[shot_index]]
    j = shot_index - 1

    while j >= 0:
        event = game_data.loc[j]
        decision = shooting_valid_sequence_event(event, shooting_team_id, current_period)

        if decision == "ignore":
            j -= 1
            continue

        if decision == "keep":
            sequence_rows.append(event)
            j -= 1
            continue

        if decision == "break":
            break

    sequence_df = pd.DataFrame(sequence_rows).sort_index()
    sequence_df["sequence_id"] = sequence_id
    sequence_df["sequence_key"] = str(game_data.loc[shot_index, "game_id"]) + "_" + str(sequence_id)
    sequence_df["sequence_event"] = range(1, len(sequence_df) + 1)
    sequence_df["shot_event_idx"] = game_data.loc[shot_index, "event_idx"]

    return sequence_df

def build_shot_sequences_for_game(game_data):
    game_data = game_data.sort_values("event_idx").reset_index(drop=True)

    sequences = []
    sequence_id = 1

    shot_mask = game_data["shot_event"].fillna(False).astype(bool)
    shot_indexes = game_data[shot_mask].index

    for shot_index in shot_indexes:
        sequence_df = get_shot_sequence(game_data, shot_index, sequence_id)
        if not sequence_df.empty:
            sequences.append(sequence_df)
            sequence_id += 1

    if not sequences:
        return pd.DataFrame()

    sequences_df = pd.concat(sequences, ignore_index=True)

    final_features = config["game_stats"].get("sequence_game_features", []).copy()
    for keyword in [
        "shot",
        "goal",
        "pass",
        "dribble",
        "aerial",
        "tackle",
        "interception",
        "clearance",
        "block",
        "loss_possession",
        "dispossessed",
        "turnover",
        "error",
        "ball_recovery",
    ]:
        final_features.extend([feature for feature in sequences_df.columns if keyword in feature])
    final_features.extend([
        "sequence_id",
        "sequence_key",
        "sequence_event",
        "shot_event_idx",
    ])

    final_features = list(dict.fromkeys(final_features))
    final_features = [col for col in final_features if col in sequences_df.columns]

    return sequences_df[final_features].reset_index(drop=True)

def passing_valid_sequence_event(event, passing_team_id, current_period):
    event_type = event["type"]
    outcome = event.get("outcome_type")
    event_period = event.get("period")
    same_team = event["team_id"] == passing_team_id

    if event_period != current_period:
        return "break"

    if event_type in ignore_types:
        return "ignore"

    if event_type in {"OffsideGiven", "OffsideProvoked"}:
        return "ignore"

    if event_type == "OffsidePass":
        return "keep_break" if same_team else "break"

    if not same_team:
        if event_type in opponent_pressure_types and outcome == "Unsuccessful":
            return "keep"
        if event_type == "Error":
            return "keep"
        if event_type == "Pass":
            return "break"
        return "keep_break"

    if event_type == "Error":
        return "keep"

    if event_type in {"Dispossessed", "Turnover"}:
        return "keep_break"

    if event_type in shot_types:
        return "keep_break"

    if event_type in break_types:
        return "keep_break"

    if event_type in {"Pass", "BallTouch"} and outcome == "Successful":
        return "keep"

    if event_type in {"Pass", "BallTouch"} and outcome != "Successful":
        return "keep_break"

    return "ignore"

def passing_sequence_ending(event, current_team_id, current_period):
    event_type = event["type"]
    outcome = event.get("outcome_type")
    same_team = event["team_id"] == current_team_id

    if event.get("period") != current_period:
        return "End of First Half" if current_period == "FirstHalf" else "End of Game"

    if same_team and event_type == "Goal":
        return "Goal"

    if same_team and event_type in shot_types:
        return "Shot"

    if same_team and event_type == "Pass" and outcome != "Successful":
        return "Unsuccessful Pass"

    if same_team and event_type == "BallTouch" and outcome != "Successful":
        return "Unsuccessful Ball Touch"

    if same_team and event_type in {"OffsideGiven", "OffsidePass", "OffsideProvoked"}:
        return "Offside"

    if not same_team:
        if event_type in {"Pass", "OffsidePass"}:
            return "Loss Possession"
        return event_type

    return event_type

def add_passing_sequence_end_metadata(current_rows, current_team_id, event, sequence_ending, period_changed=False):
    sequence_team = current_rows[0].get("team")
    opponent_pass_end = (
        not period_changed
        and event.get("team_id") != current_team_id
        and event.get("type") in {"Pass", "OffsidePass"}
    )

    for row in current_rows:
        row["sequence_team_id"] = current_team_id
        row["sequence_team"] = sequence_team
        row["sequence_ending"] = sequence_ending

        if period_changed:
            row["sequence_end_event_idx"] = None
            row["sequence_end_type"] = None
            row["sequence_end_outcome_type"] = None
            row["sequence_end_team_id"] = None
            row["sequence_end_team"] = None
            row["sequence_end_player_id"] = None
            row["sequence_end_player"] = None
            row["sequence_end_side"] = "system"
        elif opponent_pass_end:
            row["sequence_end_event_idx"] = None
            row["sequence_end_type"] = None
            row["sequence_end_outcome_type"] = None
            row["sequence_end_team_id"] = event.get("team_id")
            row["sequence_end_team"] = event.get("team")
            row["sequence_end_player_id"] = None
            row["sequence_end_player"] = None
            row["sequence_end_side"] = "opponent"
        else:
            row["sequence_end_event_idx"] = event.get("event_idx")
            row["sequence_end_type"] = event.get("type")
            row["sequence_end_outcome_type"] = event.get("outcome_type")
            row["sequence_end_team_id"] = event.get("team_id")
            row["sequence_end_team"] = event.get("team")
            row["sequence_end_player_id"] = event.get("player_id")
            row["sequence_end_player"] = event.get("player")
            row["sequence_end_side"] = "same_team" if event.get("team_id") == current_team_id else "opponent"

    return current_rows

def build_passing_sequences_for_game(game_data):

    game_data = game_data.sort_values(["period", "minute", "second", "event_idx"]).reset_index(drop=True)

    sequences = []
    current_rows = []
    current_team_id = None
    current_period = None
    sequence_id = 1

    for _, event in game_data.iterrows():
        event_type = event["type"]
        outcome = event.get("outcome_type")

        if current_team_id is None:
            if event_type == "Pass" and outcome == "Successful":
                current_team_id = event["team_id"]
                current_period = event["period"]

                row = event.copy()
                row["sequence_id"] = sequence_id
                current_rows = [row]
            continue

        pass_decision = passing_valid_sequence_event(
            event=event,
            passing_team_id=current_team_id,
            current_period=current_period,
        )

        if pass_decision == "keep":
            row = event.copy()
            row["sequence_id"] = sequence_id
            current_rows.append(row)
            continue

        if pass_decision == "keep_break":
            row = event.copy()
            row["sequence_id"] = sequence_id
            current_rows.append(row)

            sequence_ending = passing_sequence_ending(
                event=event,
                current_team_id=current_team_id,
                current_period=current_period,
            )

            current_rows = add_passing_sequence_end_metadata(
                current_rows=current_rows,
                current_team_id=current_team_id,
                event=event,
                sequence_ending=sequence_ending,
            )

            sequences.extend(current_rows)
            sequence_id += 1

            current_rows = []
            current_team_id = None
            current_period = None
            continue

        if pass_decision == "ignore":
            continue

        if pass_decision == "break":
            sequence_ending = passing_sequence_ending(
                event=event,
                current_team_id=current_team_id,
                current_period=current_period,
            )
            period_changed = event.get("period") != current_period

            current_rows = add_passing_sequence_end_metadata(
                current_rows=current_rows,
                current_team_id=current_team_id,
                event=event,
                sequence_ending=sequence_ending,
                period_changed=period_changed,
            )


            sequences.extend(current_rows)
            sequence_id += 1

            current_rows = []
            current_team_id = None
            current_period = None

            if event_type == "Pass" and outcome == "Successful":
                current_team_id = event["team_id"]
                current_period = event["period"]

                row = event.copy()
                row["sequence_id"] = sequence_id
                current_rows = [row]

    if current_rows:
        sequence_ending = "End of First Half" if current_period == "FirstHalf" else "End of Game"

        for row in current_rows:
            row["sequence_team_id"] = current_team_id
            row["sequence_team"] = current_rows[0].get("team")
            row["sequence_ending"] = sequence_ending
            row["sequence_end_event_idx"] = None
            row["sequence_end_type"] = None
            row["sequence_end_outcome_type"] = None
            row["sequence_end_team_id"] = None
            row["sequence_end_team"] = None
            row["sequence_end_player_id"] = None
            row["sequence_end_player"] = None
            row["sequence_end_side"] = "system"

        sequences.extend(current_rows)

    sequences_df = pd.DataFrame(sequences)
    if sequences_df.empty:
        return sequences_df

    sequences_df["sequence_event"] = sequences_df.groupby("sequence_id").cumcount() + 1
    sequences_df["sequence_key"] = sequences_df["game_id"].astype(str) + "_" + sequences_df["sequence_id"].astype(str)

    final_features = config["game_stats"].get("sequence_game_features", []).copy()
    final_features.extend([feature for feature in sequences_df.columns if "pass" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "ball_touch" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "shot" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "goal" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "offside" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "foul" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "tackle" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "interception" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "clearance" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "aerial" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "block" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "loss_possession" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "dispossessed" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "turnover" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "error" in feature])
    final_features.extend([feature for feature in sequences_df.columns if "ball_recovery" in feature])
    final_features.extend([
        "sequence_id",
        "sequence_key",
        "sequence_event",
        "sequence_team_id",
        "sequence_team",
        "sequence_ending",
        "sequence_end_event_idx",
        "sequence_end_type",
        "sequence_end_outcome_type",
        "sequence_end_team_id",
        "sequence_end_team",
        "sequence_end_player_id",
        "sequence_end_player",
        "sequence_end_side",
    ])

    final_features = list(dict.fromkeys(final_features))
    final_features = [col for col in final_features if col in sequences_df.columns]

    return sequences_df[final_features].reset_index(drop=True)


In [ ]:
# all_games = all_games[-2:]
# all_games

In [ ]:
all_shot_sequences = pd.DataFrame()
all_pass_sequences = pd.DataFrame()
processed_col = db[mongo_config["collections"]["collection_processed_events"]]

all_types = pd.DataFrame()
for game in tqdm(all_games):
    game_data = pd.DataFrame(
        list(processed_col.find({"game_id": game}, {"_id": 0}))
    )

    if game_data.empty:
        continue
    
    game_data = game_data.sort_values("event_idx").reset_index(drop=True)

    shot_sequence_df = build_shot_sequences_for_game(game_data)
    if not shot_sequence_df.empty:
        all_shot_sequences = pd.concat([all_shot_sequences, shot_sequence_df], ignore_index=True)

    pass_sequence_df = build_passing_sequences_for_game(game_data)
    if not pass_sequence_df.empty:
        all_pass_sequences = pd.concat([all_pass_sequences, pass_sequence_df], ignore_index=True)


In [ ]:
# #### Checking

In [ ]:
shot_sequence_endings = all_shot_sequences.sort_values(
    ["game_id", "sequence_id", "sequence_event"]
).groupby(["game_id", "sequence_id"]).tail(1)

print("Shot sequence final event types")
display(shot_sequence_endings["type"].value_counts(dropna=False))

print("Shot sequence final event type / outcome")
display(
    shot_sequence_endings.groupby(
        ["type", "outcome_type"],
        dropna=False,
    )
    .size()
    .sort_values(ascending=False)
)

checks = {
    "final_event_not_shot": shot_sequence_endings.query(
        "type not in ['Goal', 'SavedShot', 'MissedShots', 'ShotOnPost']"
    ),
    "missing_shot_event_flag": shot_sequence_endings.query(
        "shot_event != True"
    ),
    "bad_sequence_key_duplicates": all_shot_sequences[
        all_shot_sequences.duplicated(["sequence_key", "sequence_event"], keep=False)
    ],
}

for check_name, check_df in checks.items():
    print(f"{check_name}: {len(check_df)}")
    if not check_df.empty:
        display(
            check_df[
                [
                    "game_id",
                    "sequence_id",
                    "sequence_key",
                    "sequence_event",
                    "event_idx",
                    "team",
                    "player",
                    "type",
                    "outcome_type",
                    "shot_event",
                    "shot_result",
                    "shot_zone",
                    "shot_situation",
                    "shot_body_part",
                ]
            ].head(20)
        )

sequence_event_check = all_shot_sequences.groupby(
    ["game_id", "sequence_id"]
)["sequence_event"].agg(["min", "max", "count"])

bad_sequence_event_numbering = sequence_event_check.query("min != 1 or max != count")

print(f"bad_sequence_event_numbering: {len(bad_sequence_event_numbering)}")
display(bad_sequence_event_numbering.head(20))

print("Shot sequence event types")
display(all_shot_sequences["type"].value_counts(dropna=False))

print("Shot sequence type / outcome")
display(
    all_shot_sequences.groupby(["type", "outcome_type"], dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("Sequence length distribution")
display(
    all_shot_sequences.groupby(["game_id", "sequence_id"])
    .size()
    .describe()
)

In [ ]:
pass_sequence_endings = all_pass_sequences.drop_duplicates(["game_id", "sequence_id"])

print("Sequence endings")
display(pass_sequence_endings["sequence_ending"].value_counts(dropna=False))

print("Raw sequence end types")
display(pass_sequence_endings["sequence_end_type"].value_counts(dropna=False))

print("End type / outcome / side")
display(
    pass_sequence_endings.groupby(
        ["sequence_end_type", "sequence_end_outcome_type", "sequence_end_side"],
        dropna=False,
    )
    .size()
    .sort_values(ascending=False)
)

checks = {
    "substitution_endings": pass_sequence_endings.query(
        "sequence_end_type in ['SubstitutionOn', 'SubstitutionOff']"
    ),
    "error_endings": pass_sequence_endings.query(
        "sequence_end_type == 'Error'"
    ),
    "bad_offside_endings": pass_sequence_endings.query(
        "sequence_ending == 'Offside' and "
        "(sequence_end_type != 'OffsidePass' or sequence_end_side != 'same_team')"
    ),
    "same_team_successful_pass_endings": pass_sequence_endings.query(
        "sequence_end_type == 'Pass' and "
        "sequence_end_side == 'same_team' and "
        "sequence_end_outcome_type != 'Unsuccessful'"
    ),
}

for check_name, check_df in checks.items():
    print(f"{check_name}: {len(check_df)}")
    if not check_df.empty:
        display(
            check_df[
                [
                    "game_id",
                    "sequence_id",
                    "sequence_key",
                    "sequence_team",
                    "sequence_ending",
                    "sequence_end_event_idx",
                    "sequence_end_type",
                    "sequence_end_outcome_type",
                    "sequence_end_team",
                    "sequence_end_player",
                    "sequence_end_side",
                ]
            ].head(20)
        )

sequence_event_check = all_pass_sequences.groupby(
    ["game_id", "sequence_id"]
)["sequence_event"].agg(["min", "max", "count"])

bad_sequence_event_numbering = sequence_event_check.query("min != 1 or max != count")

print(f"bad_sequence_event_numbering: {len(bad_sequence_event_numbering)}")
display(bad_sequence_event_numbering.head(20))

loss_possession_endings = pass_sequence_endings.query(
    "sequence_ending == 'Loss Possession'"
)

print("Loss possession endings")
display(
    loss_possession_endings[
        [
            "game_id",
            "sequence_id",
            "sequence_key",
            "sequence_team",
            "sequence_ending",
            "sequence_end_type",
            "sequence_end_outcome_type",
            "sequence_end_team",
            "sequence_end_player",
            "sequence_end_side",
        ]
    ].head(20)
)

In [ ]:
pass_sequence_endings.query(
    "sequence_end_type.isna() and sequence_end_side == 'opponent'"
)["sequence_ending"].value_counts(dropna=False)

In [ ]:
pass_sequence_endings.query(
    "sequence_end_type.isna() and sequence_end_side == 'system'"
)["sequence_ending"].value_counts(dropna=False)

In [ ]:
all_pass_sequences[(all_pass_sequences.game_id == 327995) & (all_pass_sequences.sequence_id == 68)]

In [ ]:
game_data = pd.DataFrame(
    list(processed_col.find({"game_id": 327995}, {"_id": 0}))
)

In [ ]:
game_data[game_data.minute > 24].head(20)

In [ ]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]

In [ ]:
seasons = ['2009-2010', '2010-2011', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']
for season in seasons:
    schedule_games = list(db[mongo_config.get('collections').get('collection_schedule')].distinct('game_id', {'season': season}))
    raw_events = list(db[mongo_config.get('collections').get('collection_raw_events')].distinct('game_id', {'season': season}))
    processed_event = list(db[mongo_config.get('collections').get('collection_processed_events')].distinct('game_id', {'season': season}))
    team_stats = list(db[mongo_config.get('collections').get('collection_team_game_stats')].distinct('game_id', {'season': season}))
    player_stats = list(db[mongo_config.get('collections').get('collection_player_game_stats')].distinct('game_id', {'season': season}))
    shot_sequence_games = list(db[mongo_config.get('collections').get('collection_shot_sequences')].distinct('game_id', {'season': season}))
    pass_sequence_games = list(db[mongo_config.get('collections').get('collection_pass_sequences')].distinct('game_id', {'season': season}))

    print(f"Season: {season} -> Schedule: {len(schedule_games)} | Raw: {len(raw_events)} | Processed: {len(processed_event)} | Team Stas: {len(team_stats)} | Player Stats: {len(player_stats)} | Shot Sequence: {len(shot_sequence_games)} | Pass Sequence: {len(pass_sequence_games)}")

In [ ]:
# season = '2017-2018'
# processed_event = db[mongo_config.get('collections').get('collection_processed_events')].delete_many({'season': season})
# team_stats = db[mongo_config.get('collections').get('collection_team_game_stats')].delete_many({'season': season})
# player_stats = db[mongo_config.get('collections').get('collection_player_game_stats')].delete_many({'season': season})
# shot_sequence_games = db[mongo_config.get('collections').get('collection_shot_sequences')].delete_many({'season': season})
# pass_sequence_games = db[mongo_config.get('collections').get('collection_pass_sequences')].delete_many({'season': season}))
